Полезные ссылки
Урок в прозе (часть 1): https://proproprogs.ru/python_oop/python-kollekciya-slots

Урок в прозе (часть 2): https://proproprogs.ru/python_oop/python-kak-rabotaet-slots-s-property-i-pri-nasledovanii

In [27]:
class Point:
    MAX_COORD = 999
    def __init__(self, x, y):
        self.x = x
        self.y = y


pt = Point(1, 2)
print(pt.x)
print(pt.y)
pt.z = 34
print(pt.__dict__)
# мы можем обращаться к атрибутам x, y, менять их, а также создавать дополнительные в локальном пространстве атрибутов экземпляра


1
2
{'x': 1, 'y': 2, 'z': 34}


Но что если мы хотим объявить экземпляр, у экземпляров которого были бы только свойства x и y?


In [28]:
class Point2D:
    __slots__ = ('x', 'y') # разрешенные локальные свойства у экземпляров
    MAX_COORD = 999 # атрибутов класса может быть сколько угодно

    def __init__(self, x, y):
        self.x = x
        self.y = y

pt2 = Point2D(1, 2)
print(pt2.x)
print(pt2.y)
print(pt2.MAX_COORD)
pt.z = 34


1
2
999


In [29]:
print(pt2.__dict__)

AttributeError: 'Point2D' object has no attribute '__dict__'

Колекция __dict__ для такого экземпляра тоже отсутствует

Эти локальные свойства мы можем без проблем менять, но создавать новые не можем

Благодаря такому ограничению уменьшается объем занимаемый памятью

In [ ]:
pt.__sizeof__() + pt.__dict__.__sizeof__()

In [30]:
pt2.__sizeof__()

32

Еще наличие коллекции __slots__ ускоряет работу с атрибутами

In [55]:
import timeit

class Point:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def calc(self):
        self.x += 1
        del self.y
        self.y = 0


class Point2D:
    __slots__ = ('x', 'y')

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

    def calc(self):
        self.x += 1
        del self.y
        self.y = 0


pt = Point(1, 2)
pt2 = Point2D(1, 22)

t1 = timeit.timeit(pt.calc)
print(t1)
t2 = timeit.timeit(pt2.calc)
print(t2)

TypeError: Point2D.__init__() missing 1 required positional argument: 'z'

в python console разница есть

In [ ]:
class Point2D:
    __slots__ = ('x', 'y', 'length')

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.length = (x**2 + y**2)**0.5

# все работает

Но что если свойство length задать как свойство

In [2]:
class Point2D:
    __slots__ = ('x', 'y', '__length')

    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.__length = (x**2 + y**2)**0.5

    @property
    def length(self):
        return self.__length

    @length.setter
    def length(self, value):
        self.__length = value

pt = Point2D(1,2)
pt.length

2.23606797749979

Несмотря на отсутствие в __slots__ атрибута length, мы можем обращаться к нему. Так как это атрибут класса, а не атрибут экземпляра

Теперь посмтрим как это рабтает при наследовании классов

In [9]:
class Point2D:
    __slots__ = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y

class Point3D(Point2D):
    pass


pt3 = Point3D(1,2)
pt3.z = 4
pt3.x = 4
print('pt3.x', pt3.x)
pt3.__dict__


pt3.x 4


{'z': 4}

Коллекция __slots__ не проносится через наследование. Но при этом у дочернего класса в коллекции __dict__ не будет атрибутов из базового класса. Хотя при этом обращаться  кним и менять их можно

Если в дочернем классе прописать хотя бы пустую коллекцию __slots__, значит будут разрещены только атрибуты из базовог окласса

In [13]:
class Point2D:
    __slots__ = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y

class Point3D(Point2D):
    __slots__ = ()


pt3 = Point3D(1,2)
pt3.z = 4


AttributeError: 'Point3D' object has no attribute 'z'

если мы хотим позволитьв дочернем классе создать атрибут z, нжно добавить его к слотс

In [1]:
class Point2D:
    __slots__ = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y

class Point3D(Point2D):
    __slots__ = 'z',


pt3 = Point3D(1, 2)
pt3.x = 11
pt3.z = 33
print(pt3.x, pt3.y, pt3.z)
print(pt3.__dict__)

11 2 33


AttributeError: 'Point3D' object has no attribute '__dict__'